# FoodIntent: classificação de mensagens de delivery

Notebook reproduzível com TF-IDF, Logistic Regression e Multinomial Naive Bayes. O teste curado inclui a regressão observada na API: “Meu pedido está atrasado”.

> SID e suplemento do projeto são sintéticos. Estas métricas não representam clientes reais.

## 1. Carregar os dados

No Kaggle, habilite Internet nas configurações do notebook.

In [ ]:
import pandas as pd
from IPython.display import display

SID_URL = "https://huggingface.co/datasets/luigicfilho/sid/resolve/main"
PROJECT_URL = "https://raw.githubusercontent.com/herbetdesign/AI-Engineering-Generative-AI/main/foodintent-ml/data/curated"
SPLITS = {"train": "train.csv", "validation": "val.csv", "test": "test.csv"}
LABEL_MAP = {
    "track_order": "order_status", "check_order_status": "order_status",
    "cancel_order": "cancellation",
    "report_payment_failure": "payment_issue", "dispute_charge": "payment_issue",
    "request_refund": "refund_request",
}
TARGET_NAMES = ["order_status", "cancellation", "payment_issue", "refund_request", "delivery_problem"]

def load_sid(split):
    frame = pd.read_csv(f"{SID_URL}/{SPLITS[split]}")
    frame = frame.loc[frame["language"].eq("pt")].copy()
    frame["intent"] = frame["label"].map(LABEL_MAP)
    frame = frame.dropna(subset=["intent"])
    frame["origin"] = "sid_synthetic"
    return frame[["text", "intent", "origin"]].reset_index(drop=True)

def load_curated(split):
    frame = pd.read_csv(f"{PROJECT_URL}/{split}.csv")
    frame["origin"] = "project_authored_synthetic"
    return frame[["text", "intent", "origin"]]

sid_train = load_sid("train")
sid_validation = load_sid("validation")
sid_test = load_sid("test")
curated_train = load_curated("train")
curated_validation = load_curated("validation")
curated_test = load_curated("test")
train_df = pd.concat([sid_train, curated_train], ignore_index=True)
print({"sid_train": len(sid_train), "curated_train": len(curated_train), "validation": len(curated_validation), "sid_test": len(sid_test), "curated_test": len(curated_test)})
display(train_df.groupby(["origin", "intent"]).size().unstack(fill_value=0))

## 2. Treinar e selecionar

A seleção usa somente a validação curada, mais próxima da linguagem de delivery. O teste fica reservado. As frases curadas recebem peso maior no treinamento.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

def make_pipeline(classifier):
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=30000, strip_accents="unicode")), ("classifier", classifier)])

candidates = {
    "logistic_regression": make_pipeline(LogisticRegression(max_iter=1500, class_weight="balanced", random_state=42)),
    "multinomial_nb": make_pipeline(MultinomialNB(alpha=0.5)),
}
sample_weight = np.where(train_df["origin"].eq("project_authored_synthetic"), 8.0, 1.0)
scores = {}
for name, model in candidates.items():
    model.fit(train_df["text"], train_df["intent"], classifier__sample_weight=sample_weight)
    prediction = model.predict(curated_validation["text"])
    scores[name] = f1_score(curated_validation["intent"], prediction, labels=TARGET_NAMES, average="macro", zero_division=0)
    print(name, "curated validation macro F1:", round(scores[name], 4))

best_name = max(scores, key=scores.get)
best_model = candidates[best_name]
print("Modelo selecionado:", best_name)

## 3. Avaliação separada

Os testes do SID e do suplemento curado são reportados separadamente. Eles têm origens e limitações diferentes.

In [ ]:
def evaluate(model, frame, labels):
    prediction = model.predict(frame["text"])
    return prediction, {
        "rows": len(frame),
        "accuracy": accuracy_score(frame["intent"], prediction),
        "macro_f1": f1_score(frame["intent"], prediction, labels=labels, average="macro", zero_division=0),
        "classification_report": classification_report(frame["intent"], prediction, labels=labels, output_dict=True, zero_division=0),
    }

sid_labels = [label for label in TARGET_NAMES if label in set(sid_test["intent"])]
sid_prediction, sid_metrics = evaluate(best_model, sid_test, sid_labels)
curated_prediction, curated_metrics = evaluate(best_model, curated_test, TARGET_NAMES)
print("SID test:", {k: round(v, 4) for k, v in sid_metrics.items() if isinstance(v, float)})
print("Curated test:", {k: round(v, 4) for k, v in curated_metrics.items() if isinstance(v, float)})
print(classification_report(curated_test["intent"], curated_prediction, labels=TARGET_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(curated_test["intent"], curated_prediction, labels=TARGET_NAMES, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.tight_layout()
plt.show()

## 4. Conferir a falha observada e inspecionar erros

A confiança do classificador não é calibrada. Mesmo uma confiança alta pode acompanhar uma classe errada.

In [ ]:
curated_results = curated_test.copy()
curated_results["predicted_intent"] = curated_prediction
curated_results["confidence"] = best_model.predict_proba(curated_test["text"]).max(axis=1)
curated_results["correct"] = curated_results["intent"].eq(curated_results["predicted_intent"])
display(curated_results.loc[curated_results["text"].eq("Meu pedido está atrasado")])
display(curated_results.loc[~curated_results["correct"]])

## 5. Exportar artefatos

O arquivo de métricas e o pipeline podem ser baixados pelos outputs do Kaggle. Só publique depois de revisar os resultados e as limitações.

In [ ]:
import json
import joblib
from pathlib import Path

output_dir = Path("/kaggle/working")
output_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": best_model, "labels": TARGET_NAMES, "selected_model": best_name}, output_dir / "foodintent_pipeline.joblib")
metrics_output = {"selected_model": best_name, "curated_validation_macro_f1": scores, "sid_test": sid_metrics, "curated_test": curated_metrics}
(output_dir / "metrics.json").write_text(json.dumps(metrics_output, ensure_ascii=False, indent=2), encoding="utf-8")
print("Artefatos salvos em", output_dir)

## Fonte e limitações

SID: [luigicfilho/sid](https://huggingface.co/datasets/luigicfilho/sid), MIT. O cartão do dataset descreve exemplos sintéticos. O suplemento curado deste repositório também é sintético e pequeno. O erro visto na frase sobre atraso deve continuar sendo acompanhado; este conjunto de regressão não substitui avaliação independente com dados reais licenciados.